In [1]:
!pip install -U ultralytics


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.3 MB/s eta 0:00:00


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["YOLO_DISABLE_JPEG_REPAIR"] = "1"   # critical for Kaggle
os.environ["YOLO_VERBOSE"] = "false"


In [3]:
import shutil
from pathlib import Path

COMBINED = Path("/kaggle/working/rpc_yolo/combined_dataset")

if COMBINED.exists():
    shutil.rmtree(COMBINED)

print("🧹 Old combined dataset removed")


🧹 Old combined dataset removed


In [4]:
# %% [code]
from pathlib import Path
import yaml
from tqdm import tqdm
import os

In [5]:
# %% [code]
def assemble_combined_rpc_dataset():
    """
    OPTION 2 (BEST PRACTICE):
    - Original RPC data: copied once
    - Synthetic data: symlinked (zero disk)
    """

    # ===============================
    # SOURCE DATA (READ-ONLY)
    # ===============================
    ORIGINAL_IMAGES = {
        "train": Path("/kaggle/input/retail-product-checkout-dataset/train2019"),
        "val":   Path("/kaggle/input/retail-product-checkout-dataset/val2019"),
        "test":  Path("/kaggle/input/retail-product-checkout-dataset/test2019"),
    }

    ORIGINAL_LABELS = {
        "train": Path("/kaggle/input/rpc-data/rpc_yolo/yolo_dataset/labels/train"),
        "val":   Path("/kaggle/input/rpc-data/rpc_yolo/yolo_dataset/labels/val"),
        "test":  Path("/kaggle/input/rpc-data/rpc_yolo/yolo_dataset/labels/test"),
    }

    SYNTHETIC_IMAGES = Path(
        "/kaggle/input/synthetic-dataset-rpc/rpc_yolo/synthetic/images/train"
    )
    SYNTHETIC_LABELS = Path(
        "/kaggle/input/synthetic-dataset-rpc/rpc_yolo/synthetic/labels/train"
    )

    # ===============================
    # TARGET DATASET (WRITABLE)
    # ===============================
    TARGET_ROOT = Path("/kaggle/working/rpc_yolo/")

    for split in ["train", "val", "test"]:
        (TARGET_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
        (TARGET_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

    # ===============================
    # COPY ORIGINAL DATA (ONCE)
    # ===============================
    for split in ["train", "val", "test"]:
        print(f"\n📂 Copying ORIGINAL {split.upper()} images")
        for f in tqdm(ORIGINAL_IMAGES[split].glob("*.jpg"), desc=f"Images [{split}]"):
            dst = TARGET_ROOT / "images" / split / f.name
            if not dst.exists():
                dst.symlink_to(f) if split == "train" else dst.write_bytes(f.read_bytes())

        print(f"\n🏷️ Copying ORIGINAL {split.upper()} labels")
        for f in tqdm(ORIGINAL_LABELS[split].glob("*.txt"), desc=f"Labels [{split}]"):
            dst = TARGET_ROOT / "labels" / split / f.name
            if not dst.exists():
                dst.write_bytes(f.read_bytes())

    # ===============================
    # SYMLINK SYNTHETIC DATA (TRAIN ONLY)
    # ===============================
    # print("\n🧪 Symlinking SYNTHETIC TRAIN images")
    # for f in tqdm(SYNTHETIC_IMAGES.glob("*.jpg"), desc="Synthetic Images [train]"):
    #     dst = TARGET_ROOT / "images" / "train" / f.name
    #     if not dst.exists():
    #         dst.symlink_to(f)

    # print("\n🧪 Symlinking SYNTHETIC TRAIN labels")
    # for f in tqdm(SYNTHETIC_LABELS.glob("*.txt"), desc="Synthetic Labels [train]"):
    #     dst = TARGET_ROOT / "labels" / "train" / f.name
    #     if not dst.exists():
    #         dst.symlink_to(f)

    # ===============================
    # WRITE data.yaml
    # ===============================
    data_yaml = {
        "path": str(TARGET_ROOT),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "nc": 17,
        "names": [
            "alcohol", "candy", "canned_food", "chocolate", "dessert",
            "dried_food", "dried_fruit", "drink", "gum", "instant_drink",
            "instant_noodles", "milk", "personal_hygiene", "puffed_food",
            "seasoner", "stationery", "tissue",
        ],
    }

    with open(TARGET_ROOT / "data.yaml", "w") as f:
        yaml.dump(data_yaml, f, sort_keys=False)

   

    yaml_path = "/kaggle/working/rpc_yolo/data.yaml"
    
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    
    print(data)

    print("\n✅ Combined dataset assembled via symlinks")
    print("📁 Dataset root:", TARGET_ROOT)


In [6]:
# %% [code]
assemble_combined_rpc_dataset()

# -------------------------------
# VERIFY SYMLINKS (IMPORTANT)
# -------------------------------
train_img_dir = Path("/kaggle/working/rpc_yolo/images/train")

symlinks = sum(os.path.islink(train_img_dir / f) for f in os.listdir(train_img_dir))
total = len(os.listdir(train_img_dir))

print(f"\n🔍 Verification:")
print(f"  Total train images: {total}")
print(f"  Symlinked images:   {symlinks}")
print("✅ Setup looks correct")



📂 Copying ORIGINAL TRAIN images


Images [train]: 53739it [00:04, 13150.32it/s]



🏷️ Copying ORIGINAL TRAIN labels


Labels [train]: 53739it [05:21, 167.40it/s]



📂 Copying ORIGINAL VAL images


...


📂 Copying ORIGINAL TEST images


Images [test]: 24000it [03:01, 132.04it/s]



🏷️ Copying ORIGINAL TEST labels


Labels [test]: 24000it [02:16, 175.77it/s]


{'path': '/kaggle/working/rpc_yolo', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'nc': 17, 'names': ['alcohol', 'candy', 'canned_food', 'chocolate', 'dessert', 'dried_food', 'dried_fruit', 'drink', 'gum', 'instant_drink', 'instant_noodles', 'milk', 'personal_hygiene', 'puffed_food', 'seasoner', 'stationery', 'tissue']}

✅ Combined dataset assembled via symlinks
📁 Dataset root: /kaggle/working/rpc_yolo

🔍 Verification:
  Total train images: 53739
  Symlinked images:   53739
✅ Setup looks correct


In [7]:
# from pathlib import Path

# IMG_DIR = Path("/kaggle/input/retail-product-checkout-dataset/train2019")
# LBL_DIR = Path("/kaggle/input/rpc-data/rpc_yolo/yolo_dataset/labels/train")
# images = {p.stem for p in IMG_DIR.glob("*.jpg")}
# labels = {p.stem for p in LBL_DIR.glob("*.txt")}

# matched = images & labels
# missing_labels = images - labels
# missing_images = labels - images

# print("📊 DATASET CONSISTENCY CHECK (TRAIN)")
# print(f"Total images        : {len(images)}")
# print(f"Total labels        : {len(labels)}")
# print(f"Images with labels  : {len(matched)}")
# print(f"Images w/o labels   : {len(missing_labels)}")
# print(f"Labels w/o images   : {len(missing_images)}")


In [8]:
# from pathlib import Path

# IMG_DIR = Path("/kaggle/working/rpc_yolo/combined_dataset/images/train")
# LBL_DIR = Path("/kaggle/working/rpc_yolo/combined_dataset/labels/train")
# images = {p.stem for p in IMG_DIR.glob("*.jpg")}
# labels = {p.stem for p in LBL_DIR.glob("*.txt")}

# matched = images & labels
# missing_labels = images - labels
# missing_images = labels - images

# print("📊 DATASET CONSISTENCY CHECK (TRAIN)")
# print(f"Total images        : {len(images)}")
# print(f"Total labels        : {len(labels)}")
# print(f"Images with labels  : {len(matched)}")
# print(f"Images w/o labels   : {len(missing_labels)}")
# print(f"Labels w/o images   : {len(missing_images)}")


In [9]:
# from pathlib import Path

# LBL_DIR = Path("/kaggle/working/rpc_yolo/combined_dataset/labels/train")

# valid = 0
# empty = 0
# invalid = 0

# for lbl in LBL_DIR.glob("*.txt"):
#     lines = lbl.read_text().strip().splitlines()
    
#     if len(lines) == 0:
#         empty += 1
#         continue

#     ok = True
#     for line in lines:
#         parts = line.split()
#         if len(parts) != 5:
#             ok = False
#             break

#         cls, x, y, w, h = parts
#         try:
#             cls = int(cls)
#             x, y, w, h = map(float, (x, y, w, h))
#         except:
#             ok = False
#             break

#         if not (0 <= cls < 17):
#             ok = False
#             break
#         if not (0 < x <= 1 and 0 < y <= 1 and 0 < w <= 1 and 0 < h <= 1):
#             ok = False
#             break

#     if ok:
#         valid += 1
#     else:
#         invalid += 1

# print("📊 LABEL QUALITY CHECK (TRAIN)")
# print(f"Valid labels   : {valid}")
# print(f"Empty labels   : {empty}")
# print(f"Invalid labels : {invalid}")
# print(f"Total labels   : {valid + empty + invalid}")


In [10]:
# from pathlib import Path

# lbl_dir = Path("/kaggle/working/rpc_yolo/combined_dataset/labels/train")

# empty = [f for f in lbl_dir.glob("*.txt") if f.stat().st_size == 0]
# print("Empty label files:", len(empty))


In [11]:
from ultralytics import YOLO
from pathlib import Path
import json
import pandas as pd


In [12]:
# import cv2
# from pathlib import Path
# from tqdm import tqdm

# IMG_DIR = Path("/kaggle/working/rpc_yolo/combined_dataset/images/train")
# LBL_DIR = Path("/kaggle/working/rpc_yolo/combined_dataset/labels/train")

# bad_images = []

# for img_path in tqdm(list(IMG_DIR.glob("*.jpg")), desc="Checking images"):
#     img = cv2.imread(str(img_path))
#     if img is None:
#         bad_images.append(img_path)

# print(f"\n❌ Corrupt images found: {len(bad_images)}")


In [13]:
# for img_path in bad_images:
#     lbl_path = LBL_DIR / (img_path.stem + ".txt")
#     img_path.unlink(missing_ok=True)
#     lbl_path.unlink(missing_ok=True)

# print("✅ Corrupt images and labels removed")


In [14]:
import time

epoch_times = []

def on_train_start(trainer):
    trainer._epoch_start_time = time.time()
    print("🚀 Training started...\n")

def on_train_epoch_start(trainer):
    trainer._epoch_start_time = time.time()

def on_train_epoch_end(trainer):
    epoch_time = time.time() - trainer._epoch_start_time
    epoch_times.append(epoch_time)

    avg_epoch = sum(epoch_times) / len(epoch_times)
    remaining = trainer.epochs - (trainer.epoch + 1)
    eta = int(avg_epoch * remaining)

    h, rem = divmod(eta, 3600)
    m, s = divmod(rem, 60)

    mtr = trainer.metrics  # ← dict

    box = mtr.get("val/box_loss", float("nan"))
    cls = mtr.get("val/cls_loss", float("nan"))
    dfl = mtr.get("val/dfl_loss", float("nan"))

    map50 = mtr.get("metrics/mAP50(B)", float("nan"))

    print(
        f"Epoch {trainer.epoch+1}/{trainer.epochs} | "
        f"Time: {epoch_time:.1f}s | "
        f"ETA: {h:02d}:{m:02d}:{s:02d} | "
        f"box: {box:.4f} | "
        f"cls: {cls:.4f} | "
        f"dfl: {dfl:.4f} | "
        f"mAP50: {map50:.4f}"
    )




In [15]:
EXPERIMENTS = {
    "exp3_synthetic": {
        "name": "Synthetic Multi-Object Training",
        "description": "Train on original data",
        "data_yaml": "/kaggle/working/rpc_yolo/data.yaml",
        "model": "yolov8l.pt",
        "epochs": 35,
        "imgsz": 640,
        "batch": 12,
        "mosaic": 0.5,
        "copy_paste": 0.1,
    }
}


def run_experiment(exp_name, config):
    print("=" * 80)
    print(f"EXPERIMENT: {config['name']}")
    print("=" * 80)
    print(f"Description: {config['description']}\n")

    model = YOLO(config["model"])
    

    model.add_callback("on_train_start", on_train_start)
    model.add_callback("on_train_epoch_start", on_train_epoch_start)
    model.add_callback("on_train_epoch_end", on_train_epoch_end)

    train_args = {
        "data": config["data_yaml"],
        "epochs": config["epochs"],
        "imgsz": config["imgsz"],
        "batch": config["batch"],

        # Kaggle-safe
        "workers": 2,
        "cache": False,
        "rect": False,

        # Optimizer
        "optimizer": "AdamW",
        "lr0": 1e-3,
        "lrf": 0.01,
        "weight_decay": 5e-4,

        # Augmentation
        "mosaic": config["mosaic"],
        "copy_paste": config["copy_paste"],
        "close_mosaic": 10,

        # Training strategy
        "freeze": 10,
        "amp": True,

        # Saving
        "save": True,
        "save_period": 1,
        "plots": False,
        "verbose": True,
    }

    results = model.train(**train_args)

    
    # ===== Read epoch-wise metrics =====
    csv_path = Path(model.trainer.save_dir) / "results.csv"
    
    df = pd.read_csv(csv_path)

    last = df.iloc[-1]

    print(
        f"\nFINAL EPOCH {int(last['epoch'])}\n"
        f"mAP50      : {last['metrics/mAP50(B)']:.4f}\n"
        f"mAP50-95   : {last['metrics/mAP50-95(B)']:.4f}\n"
        f"Precision  : {last['metrics/precision(B)']:.4f}\n"
        f"Recall     : {last['metrics/recall(B)']:.4f}"
    )

    return last.to_dict()


def main():
    all_results = {}

    for exp_name, config in EXPERIMENTS.items():
        all_results[exp_name] = run_experiment(exp_name, config)

    with open("/kaggle/working/all_experiments_results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    print("\nTRAINING COMPLETE ✅")


if __name__ == "__main__":
    main()


EXPERIMENT: Synthetic Multi-Object Training
Description: Train on original data

🚀 Training started...

Epoch 1/35 | Time: 82.3s | ETA: 00:46:39 | box: 0.0000 | cls: 0.0000 | dfl: 0.0000 | mAP50: 0.0000
Epoch 2/35 | Time: 78.0s | ETA: 00:44:04 | box: 1.3868 | cls: 6.7485 | dfl: 1.9539 | mAP50: 0.0224
Epoch 3/35 | Time: 77.1s | ETA: 00:42:12 | box: 1.4218 | cls: 6.8436 | dfl: 1.7957 | mAP50: 0.0285
Epoch 4/35 | Time: 77.1s | ETA: 00:40:37 | box: 1.4112 | cls: 7.0992 | dfl: 1.8069 | mAP50: 0.0272
Epoch 5/35 | Time: 77.1s | ETA: 00:39:09 | box: 1.3660 | cls: 7.1432 | dfl: 1.7482 | mAP50: 0.0278
Epoch 6/35 | Time: 77.0s | ETA: 00:37:45 | box: 1.4860 | cls: 7.0168 | dfl: 1.8176 | mAP50: 0.0286
Epoch 7/35 | Time: 77.0s | ETA: 00:36:22 | box: 1.6014 | cls: 7.3465 | dfl: 2.0139 | mAP50: 0.0217
Epoch 8/35 | Time: 77.0s | ETA: 00:35:01 | box: 1.6270 | cls: 7.2650 | dfl: 2.0099 | mAP50: 0.0257
Epoch 9/35 | Time: 76.9s | ETA: 00:33:40 | box: 1.4646 | cls: 7.0366 | dfl: 1.7621 | mAP50: 0.0312
Epoch

In [16]:
from pathlib import Path
import shutil

run_dir = Path("runs/detect")
latest = sorted(run_dir.glob("train*"))[-1]

shutil.copy(latest / "weights/best.pt", "/kaggle/working/best.pt")
shutil.copy(latest / "weights/last.pt", "/kaggle/working/last.pt")

print("✅ Models copied to /kaggle/working")


✅ Models copied to /kaggle/working


In [17]:
# def main():
#     all_results = {}

#     for exp_name, config in EXPERIMENTS.items():
#         try:
#             all_results[exp_name] = run_experiment(exp_name, config)
#         except Exception as e:
#             print(f"✗ {exp_name} failed: {e}")

#     with open("/kaggle/working/all_experiments_results.json", "w") as f:
#         json.dump(all_results, f, indent=2)

#     print("\n" + "=" * 80)
#     print("FINAL EXPERIMENT SUMMARY")
#     print("=" * 80)

#     for exp, res in all_results.items():
#         print(
#             f"{exp:<25} | "
#             f"mAP50: {res['mAP50']:.4f} | "
#             f"mAP50-95: {res['mAP50-95']:.4f} | "
#             f"Recall: {res['recall']:.4f}"
#         )


# if __name__ == "__main__":
#     main()